# Capstone — CTR / Engagement Opportunity Scoring

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HIS-USERNAME/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb)

**Lane 4.** This notebook mirrors the deployed research paper section by section and regenerates every
number it quotes, so a reader can check the paper against running code rather than against my memory.

📄 **Paper:** https://his-username.github.io/flyrank-ml-internship/
📦 **Repository:** https://github.com/HIS-USERNAME/flyrank-ml-internship

> Skills: `skills/writing-research-papers/SKILL.md` + `skills/deploying-static-pages/SKILL.md`.
> Data: FlyRank ML Internship dataset — [flyrank.ai](https://flyrank.ai). Seed 42 throughout.

**The one-line version:** a four-line arithmetic rule, frozen before any model was fitted, reaches
**p@50 = 0.92** against a 0.254 base rate; four supervised models trained on top of it **do not beat it
at the head of the ranking** and only win at K = 200, AP and AUC — all within one fold-standard-deviation.
The model earns its place on **coverage, not precision**. And **42% of flagged pages stop under-capturing
within 30 days with nobody touching them**, which is why nothing here carries a value estimate and nothing
here is automated.

In [1]:
import os, sys, json, subprocess, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold, KFold
from sklearn.metrics import average_precision_score, roc_auc_score

SEED = 42
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/HIS-USERNAME/flyrank-ml-internship"   # <-- change to YOUR fork
CSV_REL  = "data/raw/content_refresh_anonymized.csv"
if IN_COLAB and not Path(CSV_REL).exists():
    if not os.path.isdir("flyrank-ml-internship"):
        subprocess.run(["git","clone","--depth","1",REPO_URL,"flyrank-ml-internship"], check=True)
    os.chdir("flyrank-ml-internship")
else:
    here = Path.cwd()
    for c in [here, *here.parents]:
        if (c / CSV_REL).exists():
            os.chdir(c); break
assert Path(CSV_REL).exists()
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
print(f"sklearn {sklearn.__version__} | pandas {pd.__version__} | seed {SEED}")

sklearn 1.8.0 | pandas 3.0.2 | seed 42


## 1. Question

> **Which pages that already have search visibility are converting that visibility into clicks at a rate
> below what their own exposure level would predict — and which of those are worth a human hour?**

Not *"which page gets the fewest clicks?"* — that question has a trivial and useless answer: the pages
nobody sees. A page with 40 impressions and 1 click is not a problem; it is a page nobody found. A page
with 12,000 impressions and 30 clicks **is already winning the hard part** — it ranks, it appears, people
see it — and is losing the cheap part, the part a title rewrite fixes in an hour.

**The decision this supports:** a content lead has one reviewer-week and an inventory in the tens of
thousands. They need an ordered list of maybe 200 pages, each with a reason they can argue with, and an
honest statement of how often the list will be wrong.

**Why the question is answerable at all:** the release carries a prior 30-day window and a later 30-day
window on the same pages. That means an outcome that happens *after* the features are observed — the
minimum requirement for a supervised claim rather than a description.

In [2]:
MIN_IMPRESSIONS, N_BANDS, BOTTOM_QUANTILE = 500, 4, 0.25
df = pd.read_csv(CSV_REL)

# QUEUE = everything scoreable at the decision moment (prior window only)
q = df[df["impressions_prev_30d"] >= MIN_IMPRESSIONS].copy()
q["ctr_prev"] = q["clicks_prev_30d"] / q["impressions_prev_30d"] * 100
q["band_prev"] = pd.qcut(q["impressions_prev_30d"], N_BANDS, labels=False, duplicates="drop")
q["peer_median_ctr"] = q.groupby("band_prev")["ctr_prev"].transform("median")
q["ctr_shortfall_pp"] = (q["peer_median_ctr"] - q["ctr_prev"]).clip(lower=0)
q["baseline_score"] = q["ctr_shortfall_pp"] * np.log1p(q["impressions_prev_30d"])
q["ctr_pctile_prev"] = q.groupby("band_prev")["ctr_prev"].rank(pct=True)
q["has_word_count"] = q["word_count"].notna().astype(int)
q["has_keyword_data"] = q["search_volume"].notna().astype(int)
q = q.reset_index(drop=True)

# EVAL = the subset where the later-window outcome is measurable
ev = q[q["impressions_last_30d"] >= MIN_IMPRESSIONS].copy()
ev["ctr_last"] = ev["clicks_last_30d"] / ev["impressions_last_30d"] * 100
ev["band_last"] = pd.qcut(ev["impressions_last_30d"], N_BANDS, labels=False, duplicates="drop")
ev["y"] = (ev.groupby("band_last")["ctr_last"].rank(pct=True) <= BOTTOM_QUANTILE).astype(int)
ev = ev.reset_index(drop=True)
BASE_RATE = ev["y"].mean()

print(f"release rows            : {len(df):,} pages / {df['client_id'].nunique()} clients")
print(f"queue (>= {MIN_IMPRESSIONS} impr prev): {len(q):,} pages / {q['client_id'].nunique()} clients")
print(f"eval (outcome visible)  : {len(ev):,} pages / {ev['client_id'].nunique()} clients "
      f"({len(ev)/len(q)*100:.1f}% coverage)")
print(f"base rate (P(y=1))      : {BASE_RATE:.4f}")
print(f"\ndropped from queue to eval: {len(q)-len(ev):,} pages that fell below {MIN_IMPRESSIONS}")
print("impressions in the later window. They are NOT unmeasurable noise - they are pages that")
print("lost visibility, and excluding them is the single biggest population caveat in this work.")

release rows            : 30,000 pages / 32 clients
queue (>= 500 impr prev): 11,119 pages / 28 clients
eval (outcome visible)  : 8,825 pages / 28 clients (79.4% coverage)
base rate (P(y=1))      : 0.2544

dropped from queue to eval: 2,294 pages that fell below 500
impressions in the later window. They are NOT unmeasurable noise - they are pages that
lost visibility, and excluding them is the single biggest population caveat in this work.


## 2. Data

**Release:** `data/raw/content_refresh_anonymized.csv` — the anonymized starter release of the FlyRank
ML Internship dataset. 30,000 rows, 44 columns, 32 pseudonymized clients. Every identifier is a pseudonym
(`content_id`, `client_id`); no domains, URLs, page titles or raw search queries exist anywhere in it.

**The two windows.** `*_prev_30d` is what a reviewer could see on the decision day.
`*_last_30d` is what happened afterwards. Nothing from the later window enters a feature. Ever.

**The exclusions, and what they cost:**

| Filter | Kept | Why | What it costs |
|---|---|---|---|
| `impressions_prev_30d ≥ 500` | 11,119 pages, 28 clients | Below this, CTR is a ratio of small integers and moves ±100% on one click | Silently removes 4 clients from the study |
| `impressions_last_30d ≥ 500` | 8,825 pages | Outcome must be measurable on the same footing | 2,294 pages (20.6%) — **these are pages that lost visibility, a real outcome I do not model** |

**Columns refused as features**, all of them 90-day aggregates that overlap the outcome window:
`ctr`, `avg_position`, everything ending `_90d`, `position_tier`, `impression_tier`. §3 measures what
including one of them would have bought.

**Two gotchas the release documents and I obeyed:** rate columns are ×100 percentages, not fractions;
`avg_position = 0` means *no data*, not *rank zero*.

In [3]:
BANNED = [c for c in df.columns if c.endswith("_90d")] + \
         ["ctr","avg_position","position_tier","impression_tier","is_declining_label","trend_direction"]
BANNED = [c for c in BANNED if c in df.columns]
print(f"columns refused as features ({len(BANNED)}):")
for c in BANNED: print(f"  {c}")

print("\nCLIENT ATTRITION FROM THE FILTERS")
print(f"  release      : {df['client_id'].nunique()} clients")
print(f"  queue        : {q['client_id'].nunique()} clients")
print(f"  eval         : {ev['client_id'].nunique()} clients")
lost = sorted(set(df['client_id']) - set(q['client_id']))
print(f"  dropped by the impression floor: {len(lost)} clients -> {lost}")

print("\nTHE PAGES THE EVAL FILTER REMOVES (visibility loss, unmodelled)")
gone = q[q["impressions_last_30d"] < MIN_IMPRESSIONS]
print(f"  n = {len(gone):,} ({len(gone)/len(q)*100:.1f}% of the queue)")
print(f"  median impressions prev : {gone['impressions_prev_30d'].median():,.0f}")
print(f"  median impressions last : {gone['impressions_last_30d'].median():,.0f}")
print("  A page going from real volume to near-zero is a bigger problem than a low CTR.")
print("  This study cannot see it, and the paper says so in section 5.")

print("\nGOTCHA CHECKS")
print(f"  ctr_prev range: {q['ctr_prev'].min():.2f} .. {q['ctr_prev'].max():.2f}  "
      f"(percent scale confirmed, not 0-1)")
if "avg_position" in df.columns:
    print(f"  avg_position == 0 rows: {(df['avg_position']==0).sum():,}  "
          f"(treated as MISSING, never as rank 0)")

columns refused as features (13):
  impressions_90d
  clicks_90d
  pageviews_90d
  sessions_90d
  users_90d
  engaged_sessions_90d
  ai_sessions_90d
  scroll_events_90d
  ctr
  avg_position
  position_tier
  impression_tier
  trend_direction

CLIENT ATTRITION FROM THE FILTERS
  release      : 32 clients
  queue        : 28 clients
  eval         : 28 clients
  dropped by the impression floor: 4 clients -> ['client_1a6562590e', 'client_25fc0e7096', 'client_98a3ab7c34', 'client_e29c9c180c']

THE PAGES THE EVAL FILTER REMOVES (visibility loss, unmodelled)
  n = 2,294 (20.6% of the queue)
  median impressions prev : 724
  median impressions last : 323
  A page going from real volume to near-zero is a bigger problem than a low CTR.
  This study cannot see it, and the paper says so in section 5.

GOTCHA CHECKS
  ctr_prev range: 0.00 .. 4.31  (percent scale confirmed, not 0-1)
  avg_position == 0 rows: 1,205  (treated as MISSING, never as rank 0)


## 3. Methodology

### Task type and target

**Supervised binary classification, used as a ranker.** The target is
`under_next30 = 1` if a page sits in the **bottom quartile of CTR within its later-window volume band**.

Banding the outcome matters. Without it, "worst CTR" just re-selects the highest-volume pages, because CTR
falls with exposure. Ranking *within* a volume band asks the question I actually mean:
*is this page underperforming for a page like it?*

**The metric is precision@K, not accuracy.** A reviewer works the top of the list and never sees the rest.
Accuracy on 8,825 pages is a number about the 8,625 pages nobody will look at.

### The baseline, frozen before any model

```
ctr_shortfall_pp = max(0, peer_band_median_ctr − page_ctr)
baseline_score   = ctr_shortfall_pp × log1p(impressions_prev_30d)
```

Four lines. Multiplying shortfall by log-volume says *"a big gap on a page nobody sees is not urgent."*
It was written, scored, and **committed to `work/outputs/baseline_metrics.json` before a single model
was fitted** — so the comparison in §4 is a real test rather than a story told afterwards.

### Validation design

`GroupKFold(n_splits=5)` on `client_id`. A random split lets the same client appear in train and test,
and the model then partly recognises *whose site this is* instead of *what an under-capturing page looks
like*. §4 measures exactly how much that inflates the score.

### Leakage checks

Three: (1) window separation — no `*_last_30d` column is a feature; (2) the strongest excluded column
scored solo, to price what the discipline costs; (3) client identity — can a model recover `client_id`
from the features alone?

In [4]:
def p_at_k(y_true, score, k):
    k = min(k, len(y_true))
    return np.asarray(y_true)[np.argsort(-np.asarray(score), kind="stable")[:k]].mean()

FROZEN = {"p@10":1.000, "p@20":1.000, "p@50": 0.920, "p@200": 0.820}
repro  = {f"p@{k}": round(p_at_k(ev["y"], ev["baseline_score"], k), 3) for k in (10,20,50,200)}
assert repro == FROZEN, f"the ML-07 baseline drifted: {repro}"
print("BASELINE, REPRODUCED FROM THE FROZEN ML-07 RECEIPT")
for k, v in repro.items():
    print(f"  {k:6s} {v:.3f}   ({v/BASE_RATE:.1f}x the {BASE_RATE:.3f} base rate)")
print("\n  assert passed - if any of these had drifted this cell would have raised.")

NUMERIC = ["impressions_prev_30d","clicks_prev_30d","ctr_prev","ctr_pctile_prev","sessions_prev_30d",
           "content_age_days","days_since_last_update","word_count","char_count",
           "search_volume","competition","cpc","has_word_count","has_keyword_data"]
CATEGORICAL = ["content_type","main_intent","competition_level","age_tier","freshness_tier",
               "word_count_tier"]
FEATURES = NUMERIC + CATEGORICAL
pre = ColumnTransformer([
    ("num", Pipeline([("i",SimpleImputer(strategy="median")),("s",StandardScaler())]), NUMERIC),
    ("cat", Pipeline([("i",SimpleImputer(strategy="constant",fill_value="unknown")),
                      ("o",OneHotEncoder(handle_unknown="ignore",min_frequency=20))]), CATEGORICAL)])
print(f"\nfeature set: {len(NUMERIC)} numeric + {len(CATEGORICAL)} categorical")
print(f"no feature drawn from the later window: "
      f"{all('last_30d' not in f for f in FEATURES)}")

BASELINE, REPRODUCED FROM THE FROZEN ML-07 RECEIPT
  p@10   1.000   (3.9x the 0.254 base rate)
  p@20   1.000   (3.9x the 0.254 base rate)
  p@50   0.920   (3.6x the 0.254 base rate)
  p@200  0.820   (3.2x the 0.254 base rate)

  assert passed - if any of these had drifted this cell would have raised.

feature set: 14 numeric + 6 categorical
no feature drawn from the later window: True


## 4. Results (vs the baseline)

The four models below are fitted out-of-fold under `GroupKFold` on `client_id` and scored against the
**same** evaluation slice as the frozen rule. Nothing is tuned against the test folds.

In [5]:
MODELS = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=SEED),
    "Decision Tree (d=3)": DecisionTreeClassifier(max_depth=3, random_state=SEED),
    "Random Forest":       RandomForestClassifier(n_estimators=400, min_samples_leaf=5,
                                                  random_state=SEED, n_jobs=-1),
    "Gradient Boosting":   HistGradientBoostingClassifier(random_state=SEED),
}
X, y, g = ev[FEATURES], ev["y"].values, ev["client_id"].values
gkf = list(GroupKFold(n_splits=5).split(X, y, g))

oof_scores, per_fold = {}, {}
for name, est in MODELS.items():
    oof = np.zeros(len(ev)); folds = []
    for tr, te in gkf:
        m = Pipeline([("p",pre),("m",est)]).fit(X.iloc[tr], y[tr])
        oof[te] = m.predict_proba(X.iloc[te])[:,1]
        folds.append(p_at_k(y[te], oof[te], 50))
    oof_scores[name] = oof
    per_fold[name] = (float(np.mean(folds)), float(np.std(folds, ddof=1)))

oof_scores["Baseline rule (frozen)"] = ev["baseline_score"].values
_bf = [p_at_k(y[te], ev["baseline_score"].values[te], 50) for _, te in gkf]
per_fold["Baseline rule (frozen)"] = (float(np.mean(_bf)), float(np.std(_bf, ddof=1)))
rng = np.random.default_rng(SEED)
oof_scores["Random shortlist (base rate)"] = rng.random(len(ev))

rows = []
for name, s in oof_scores.items():
    rows.append({"model": name,
                 "p@10": round(p_at_k(y, s, 10), 3), "p@20": round(p_at_k(y, s, 20), 3),
                 "p@50": round(p_at_k(y, s, 50), 3), "p@200": round(p_at_k(y, s, 200), 3),
                 "AP": round(average_precision_score(y, s), 3),
                 "AUC": round(roc_auc_score(y, s), 3),
                 "p@50 fold mean±std": (f"{per_fold[name][0]:.3f} ± {per_fold[name][1]:.3f}"
                                        if name in per_fold else "-")})
res = pd.DataFrame(rows).sort_values("AP", ascending=False)
print("POOLED OUT-OF-FOLD, GroupKFold on client_id, same eval slice\n")
print(res.to_string(index=False))
print(f"\nbase rate = {BASE_RATE:.3f}")

POOLED OUT-OF-FOLD, GroupKFold on client_id, same eval slice

                       model  p@10  p@20  p@50  p@200    AP   AUC p@50 fold mean±std
         Logistic Regression   0.8  0.85  0.84  0.875 0.617 0.837      0.824 ± 0.091
               Random Forest   0.9  0.95  0.92  0.825 0.615 0.832      0.808 ± 0.115
           Gradient Boosting   1.0  0.95  0.86  0.870 0.593 0.818      0.796 ± 0.086
      Baseline rule (frozen)   1.0  1.00  0.92  0.820 0.580 0.803      0.804 ± 0.107
         Decision Tree (d=3)   0.3  0.65  0.64  0.720 0.545 0.817      0.612 ± 0.101
Random shortlist (base rate)   0.1  0.15  0.18  0.235 0.254 0.501                  -

base rate = 0.254


### What that table says, stated carefully

**The rule is not beaten at the head of the ranking.** At K = 10, 20 and 50 the frozen four-line rule
matches or exceeds every model. Random Forest ties it at p@50 = 0.92. Gradient Boosting matches it at
K = 10 and loses at K = 50.

**The models win further down and on the aggregate metrics** — p@200, AP, AUC — but every one of those
gaps is **smaller than one fold-standard-deviation** (±0.09 to ±0.12 on p@50 across five client folds).
On the evidence available I cannot say a model is better. I can say they are comparable.

**Why the rule is hard to beat:** the target is *bottom quartile of banded CTR later*, and the rule is
*distance below the banded CTR median now*. Those are close to the same construct one window apart. The
rule is not naive — it encodes the right idea, and the models are being asked to improve on a good idea
using features that are mostly proxies for it.

**Where the models earn their place:** coverage. The rule assigns `baseline_score = 0` to every page at or
above its peer median, and the log-volume multiplier pushes the rest down. The measured result:
**the rule's top 200 contains 0 pages from volume bands 0–2; the model's contains 44** — and 3,195 of
those 6,053 pages sit at exactly zero, tied, so the rule cannot even order them. Those pages under-capture
at 0.256, the same rate as the population, so they are not safe to ignore. Putting a real score on them is
a different job, not a better score.

In [6]:
# Disagreement: where do the two rankings differ, and who is right there?
rule_top = set(np.argsort(-ev["baseline_score"].values, kind="stable")[:200])
lr_top   = set(np.argsort(-oof_scores["Logistic Regression"], kind="stable")[:200])
shared   = rule_top & lr_top
model_only, rule_only = lr_top - rule_top, rule_top - lr_top
print("TOP-200 DISAGREEMENT (frozen rule vs Logistic Regression)\n")
print(f"  shared by both      : {len(shared):3d} pages")
print(f"  model only          : {len(model_only):3d} pages | precision "
      f"{y[list(model_only)].mean():.4f}")
print(f"  rule only           : {len(rule_only):3d} pages | precision "
      f"{y[list(rule_only)].mean():.4f}")
print(f"\n  On their {len(model_only)} points of disagreement the model is more often right")
print("  (0.837 vs 0.724). That is the honest case for running it - on a subset of ~100 pages,")
print("  which is not enough to call the difference established.")

print("\nCOVERAGE, THE ACTUAL ARGUMENT")
low = (ev["band_prev"] < 3).values
rule_order = np.argsort(-ev["baseline_score"].values, kind="stable")
lr_order   = np.argsort(-oof_scores["Logistic Regression"], kind="stable")
print(f"  pages in volume bands 0-2           : {int(low.sum()):,}")
print(f"  of those, rule score == 0           : {int((ev['baseline_score'].values[low]==0).sum()):,} "
      f"({(ev['baseline_score'].values[low]==0).mean()*100:.0f}% - tied at zero, unrankable)")
print(f"  their true under-capture rate       : {y[low].mean():.3f}  (same as the population)")
print(f"  bands 0-2 pages in the RULE  top 200: {int(low[rule_order[:200]].sum()):3d}")
print(f"  bands 0-2 pages in the MODEL top 200: {int(low[lr_order[:200]].sum()):3d}")
print("  The log-volume multiplier pushes low-volume pages down by construction, and half of")
print("  them are tied at exactly zero so the rule cannot order them at all. The model can.")

TOP-200 DISAGREEMENT (frozen rule vs Logistic Regression)

  shared by both      : 102 pages
  model only          :  98 pages | precision 0.8367
  rule only           :  98 pages | precision 0.7245

  On their 98 points of disagreement the model is more often right
  (0.837 vs 0.724). That is the honest case for running it - on a subset of ~100 pages,
  which is not enough to call the difference established.

COVERAGE, THE ACTUAL ARGUMENT
  pages in volume bands 0-2           : 6,053
  of those, rule score == 0           : 3,195 (53% - tied at zero, unrankable)
  their true under-capture rate       : 0.256  (same as the population)
  bands 0-2 pages in the RULE  top 200:   0
  bands 0-2 pages in the MODEL top 200:  44
  The log-volume multiplier pushes low-volume pages down by construction, and half of
  them are tied at exactly zero so the rule cannot order them at all. The model can.


### The split design changes the answer

Before I trusted any of the above, I ran the same two models under a **random** 5-fold split — the split
most people reach for by default — and under the grouped split.

In [7]:
def eval_split(splits, est, name):
    """ML-09 reports FOLD MEANS, not a pooled score: a pooled number hides which fold
       carried it, and the whole point of the audit is the between-fold spread."""
    f50, f200, fap, fauc = [], [], [], []
    for tr, te in splits:
        m = Pipeline([("p",pre),("m",est)]).fit(X.iloc[tr], y[tr])
        s_ = m.predict_proba(X.iloc[te])[:,1]
        f50.append(p_at_k(y[te], s_, 50)); f200.append(p_at_k(y[te], s_, 200))
        fap.append(average_precision_score(y[te], s_)); fauc.append(roc_auc_score(y[te], s_))
    return {"design": name, "p@50": round(float(np.mean(f50)),3),
            "p@200": round(float(np.mean(f200)),3), "AP": round(float(np.mean(fap)),4),
            "AUC": round(float(np.mean(fauc)),4),
            "fold spread (p@50 std)": round(float(np.std(f50, ddof=1)),4)}

rand = list(KFold(n_splits=5, shuffle=True, random_state=SEED).split(X))
rows = []
for mname, est in [("Logistic Regression", LogisticRegression(max_iter=2000, random_state=SEED)),
                   ("Random Forest", RandomForestClassifier(n_estimators=400, min_samples_leaf=5,
                                                            random_state=SEED, n_jobs=-1))]:
    for splits, label in [(rand, "BEFORE random"), (gkf, "AFTER grouped")]:
        r = eval_split(splits, est, label); r["model"] = mname; rows.append(r)
ba = pd.DataFrame(rows)[["model","design","p@50","p@200","AP","AUC","fold spread (p@50 std)"]]
print("SPLIT DESIGN: BEFORE (random) vs AFTER (grouped by client)\n")
print(ba.to_string(index=False))
print("\n  Random 5-fold inflates p@50 by ~0.05 and roughly HALVES the reported fold spread.")
print("  Both directions of that error push the same way: it makes the work look more certain")
print("  than it is. Every number in the paper is the AFTER number.")

SPLIT DESIGN: BEFORE (random) vs AFTER (grouped by client)

              model        design  p@50  p@200     AP    AUC  fold spread (p@50 std)
Logistic Regression BEFORE random 0.880  0.705 0.6302 0.8418                  0.0469
Logistic Regression AFTER grouped 0.824  0.648 0.5999 0.8339                  0.0910
      Random Forest BEFORE random 0.852  0.699 0.6298 0.8398                  0.0415
      Random Forest AFTER grouped 0.808  0.654 0.5992 0.8327                  0.1154

  Random 5-fold inflates p@50 by ~0.05 and roughly HALVES the reported fold spread.
  Both directions of that error push the same way: it makes the work look more certain
  than it is. Every number in the paper is the AFTER number.


In [8]:
# Leakage audit: what does the feature discipline cost, and is client identity leaking?
def solo_auc(col):
    v = ev[col]
    if v.dtype.kind not in "if": return None
    v = v.fillna(v.median())
    return max(roc_auc_score(y, v), roc_auc_score(y, -v))

excluded_num = [c for c in BANNED if c in ev.columns and ev[c].dtype.kind in "if"]
ex = sorted([(c, solo_auc(c)) for c in excluded_num if solo_auc(c)], key=lambda t: -t[1])
inc = sorted([(c, solo_auc(c)) for c in NUMERIC if solo_auc(c)], key=lambda t: -t[1])
print("CHECK 1 - what the excluded columns would have bought\n")
print("  strongest EXCLUDED column (solo AUC):")
for c, a in ex[:3]: print(f"    {c:28s} {a:.3f}")
print("  strongest INCLUDED column (solo AUC):")
for c, a in inc[:3]: print(f"    {c:28s} {a:.3f}")
print(f"\n  Excluding '{ex[0][0]}' costs real signal ({ex[0][1]:.3f} vs {inc[0][1]:.3f} solo).")
print("  It is excluded anyway: it is a 90-day aggregate whose window overlaps the outcome.")
print("  A model using it would score better and mean less.")

print("\nCHECK 2 - can the features recover WHICH CLIENT a page belongs to?")
big = ev["client_id"].value_counts().index[0]
yid = (ev["client_id"] == big).astype(int).values
oid = np.zeros(len(ev))
for tr, te in KFold(n_splits=5, shuffle=True, random_state=SEED).split(X):
    m = Pipeline([("p",pre),("m",LogisticRegression(max_iter=2000, random_state=SEED))]).fit(
        X.iloc[tr], yid[tr])
    oid[te] = m.predict_proba(X.iloc[te])[:,1]
print(f"  client identifiability AUC (largest client vs rest): {roc_auc_score(yid, oid):.4f}")
print("  Client identity IS strongly recoverable from the features. This is precisely why the")
print("  grouped split is mandatory - and why identity leaks into fold VARIANCE (spread doubles)")
print("  rather than into the mean: the features carry the client's scale, not its label.")

CHECK 1 - what the excluded columns would have bought

  strongest EXCLUDED column (solo AUC):
    ctr                          0.913
    clicks_90d                   0.804
    engaged_sessions_90d         0.670
  strongest INCLUDED column (solo AUC):
    ctr_pctile_prev              0.823
    ctr_prev                     0.823
    clicks_prev_30d              0.753

  Excluding 'ctr' costs real signal (0.913 vs 0.823 solo).
  It is excluded anyway: it is a 90-day aggregate whose window overlaps the outcome.
  A model using it would score better and mean less.

CHECK 2 - can the features recover WHICH CLIENT a page belongs to?
  client identifiability AUC (largest client vs rest): 0.9390
  Client identity IS strongly recoverable from the features. This is precisely why the
  grouped split is mandatory - and why identity leaks into fold VARIANCE (spread doubles)
  rather than into the mean: the features carry the client's scale, not its label.


## 5. Limitations & honest framing

These are ordered by how much they should change a reader's confidence, not by how comfortable they are.

**1. The eval slice is not the queue.** 8,825 of 11,119 pages have a measurable outcome — 79.4% coverage.
The 2,294 excluded pages fell below 500 impressions in the later window: **they lost visibility**, which is
a worse outcome than low CTR and one this study cannot model. Every precision number applies to pages that
kept their exposure.

**2. Four clients disappear before the study starts.** The impression floor removes them entirely. The
28 that remain are the higher-volume clients, and the conclusions are about them.

**3. Three of five folds test a single client.** `GroupKFold` on 28 clients with a skewed page
distribution produces folds dominated by one site. The ±0.09–0.12 fold spread is therefore partly
between-client variation, not model instability — but I cannot separate them at this sample size.

**4. The model's advantage is not established, only measured.** Every model-vs-rule gap is inside one
fold-std. I report them as *comparable*. Anyone reading "the model is better" into §4 is reading more
than the data supports.

**5. One window pair, one release.** Two 30-day windows on 30,000 rows. Seasonality, algorithm updates,
and site migrations are all invisible here. Nothing in this work has been tested on a second period.

**6. The target is a construct, not a fact.** "Bottom quartile of banded CTR" is a definition I chose.
A different band count or quantile would produce a different label and different numbers. §3 shows the
choice; it does not justify it as the only reasonable one.

**7. Under-capture is not the same as fixable.** The whole system identifies a *gap*. It contains no
evidence that a title rewrite closes that gap — and §6 shows a measurement that actively complicates the
assumption.

In [9]:
print("LIMITATION 1 - THE COVERAGE GAP, MEASURED\n")
print(f"  queue: {len(q):,}   eval: {len(ev):,}   coverage: {len(ev)/len(q)*100:.1f}%")
print(f"  excluded: {len(q)-len(ev):,} pages that lost visibility\n")

print("LIMITATION 3 - FOLD COMPOSITION\n")
for i, (tr, te) in enumerate(gkf):
    cl = pd.Series(g[te]).nunique(); top = pd.Series(g[te]).value_counts()
    print(f"  fold {i}: {len(te):5,} pages | {cl:2d} client(s) | "
          f"largest share {top.iloc[0]/len(te)*100:5.1f}%")
single = sum(1 for _, te in gkf if pd.Series(g[te]).nunique() == 1)
print(f"\n  folds testing exactly ONE client: {single} of 5")

print("\nLIMITATION 6 - THE TARGET IS A CHOICE. Sensitivity to it:\n")
for nb in (3, 4, 5):
    for bq in (0.20, 0.25, 0.30):
        t = ev.copy()
        t["b"] = pd.qcut(t["impressions_last_30d"], nb, labels=False, duplicates="drop")
        yy = (t.groupby("b")["ctr_last"].rank(pct=True) <= bq).astype(int).values
        print(f"  bands={nb} quantile={bq:.2f} -> base {yy.mean():.3f} | "
              f"rule p@50 {p_at_k(yy, ev['baseline_score'], 50):.3f} | "
              f"lift {p_at_k(yy, ev['baseline_score'], 50)/yy.mean():.1f}x")
print("\n  The rule's LIFT over the base rate is stable across all nine settings. The absolute")
print("  p@50 is not. Reporting lift alongside precision is the honest way to quote this.")

LIMITATION 1 - THE COVERAGE GAP, MEASURED

  queue: 11,119   eval: 8,825   coverage: 79.4%
  excluded: 2,294 pages that lost visibility

LIMITATION 3 - FOLD COMPOSITION

  fold 0: 3,045 pages |  1 client(s) | largest share 100.0%
  fold 1: 2,073 pages |  1 client(s) | largest share 100.0%
  fold 2: 1,239 pages |  1 client(s) | largest share 100.0%
  fold 3: 1,234 pages | 12 client(s) | largest share  44.1%
  fold 4: 1,234 pages | 13 client(s) | largest share  35.4%

  folds testing exactly ONE client: 3 of 5

LIMITATION 6 - THE TARGET IS A CHOICE. Sensitivity to it:

  bands=3 quantile=0.20 -> base 0.215 | rule p@50 0.900 | lift 4.2x
  bands=3 quantile=0.25 -> base 0.250 | rule p@50 0.920 | lift 3.7x
  bands=3 quantile=0.30 -> base 0.300 | rule p@50 0.980 | lift 3.3x
  bands=4 quantile=0.20 -> base 0.217 | rule p@50 0.900 | lift 4.1x
  bands=4 quantile=0.25 -> base 0.254 | rule p@50 0.920 | lift 3.6x
  bands=4 quantile=0.30 -> base 0.300 | rule p@50 0.980 | lift 3.3x
  bands=5 quantile

## 6. Ranked recommendations

### The finding that shapes everything: pages recover on their own

Before recommending anything, I measured what happens to a flagged page when **nobody does anything**.
The release gives a prior and a later window on the same pages, so this is directly observable.

In [10]:
# exactly the ML-10 definition: the queue-wide banded percentile, not one recomputed inside eval
ev["flagged_prior"] = (ev["ctr_pctile_prev"] <= BOTTOM_QUANTILE).astype(int)
fl, fine = ev[ev["flagged_prior"]==1], ev[ev["flagged_prior"]==0]
persist  = fl["y"].mean(); recover = 1 - persist; onset = fine["y"].mean()
print("WHAT HAPPENS WITH NO INTERVENTION AT ALL\n")
print(f"  flagged in the prior window : {len(fl):,} pages")
print(f"    still under-capturing 30d later : {persist:.3f}")
print(f"    RECOVERED UNAIDED               : {recover:.3f}   <-- {recover*100:.0f}%")
print(f"  fine in the prior window    : {len(fine):,} pages")
print(f"    newly under-capturing 30d later : {onset:.3f}")
print(f"  population base rate        : {BASE_RATE:.4f}")
print("\n  Flagging is real signal: a flagged page is "
      f"{persist/onset:.1f}x more likely to be under-capturing later than a clean one.")
print(f"  But {recover*100:.0f}% of a queue resolves itself in a month with nobody touching it.")
print("\n  CONSEQUENCE: any 'this fix earned X clicks' claim built on before/after on this queue")
print("  is inflated by roughly this rate. That is why nothing below carries a value estimate,")
print("  and why the first recommendation is a holdout rather than a rollout.")

WHAT HAPPENS WITH NO INTERVENTION AT ALL

  flagged in the prior window : 2,175 pages
    still under-capturing 30d later : 0.583
    RECOVERED UNAIDED               : 0.417   <-- 42%
  fine in the prior window    : 6,650 pages
    newly under-capturing 30d later : 0.147
  population base rate        : 0.2544

  Flagging is real signal: a flagged page is 4.0x more likely to be under-capturing later than a clean one.
  But 42% of a queue resolves itself in a month with nobody touching it.

  CONSEQUENCE: any 'this fix earned X clicks' claim built on before/after on this queue
  is inflated by roughly this rate. That is why nothing below carries a value estimate,
  and why the first recommendation is a holdout rather than a rollout.


### The recommendations, ranked by confidence

**1 — Work the top 200 of the rule-ordered queue, with a holdout.** *(high confidence)*
p@50 = 0.92 against a 0.254 base rate is a real, reproduced, frozen-in-advance result. Hold back a random
20% of the flagged pages and touch nothing on them, so the next cycle can separate the fix from the 42%
that recover regardless. **Without the holdout this system cannot ever be shown to work.**

**2 — Run the model as a second, separate list for volume bands 0–2.** *(medium confidence)*
The rule's top 200 contains 0 pages from bands 0–2 and leaves 3,195 of them tied at a score of zero;
the model's top 200 reaches 44 of them. Keep the lists
**separate and labelled** — the model's list is lower confidence and should be reviewed as such. Do not
blend the two scores; I could not explain a blended number to a reviewer.

**3 — Report lift, not just precision.** *(high confidence)*
The §5 sensitivity grid shows p@50 moves with the label definition while lift over the base rate stays
stable. Lift is the number that survives a change of definition.

**4 — Treat `ZERO_CLICKS_ON_REAL_VOLUME` as the highest-yield reason code.** *(medium confidence)*
It carries a 0.578 under-capture rate versus 0.072 for `AT_OR_ABOVE_VOLUME_PEERS` — an eight-fold
separation on a code any reviewer can verify by looking at one number.

**5 — Re-run the whole thing on a second window pair before trusting any of it operationally.**
*(stated as a requirement, not a finding)* One window pair is one observation.

In [11]:
FAR_BELOW_PP = 0.10
def reason_code(r):
    if r["clicks_prev_30d"] == 0:              return "ZERO_CLICKS_ON_REAL_VOLUME"
    if r["ctr_shortfall_pp"] >= FAR_BELOW_PP:  return "CTR_FAR_BELOW_VOLUME_PEERS"
    if r["ctr_shortfall_pp"] > 0:              return "CTR_BELOW_VOLUME_PEERS"
    return "AT_OR_ABOVE_VOLUME_PEERS"
ACTION_FOR = {"ZERO_CLICKS_ON_REAL_VOLUME":"rewrite_title_meta",
              "CTR_FAR_BELOW_VOLUME_PEERS":"rewrite_title_meta",
              "CTR_BELOW_VOLUME_PEERS":"review_intent_match",
              "AT_OR_ABOVE_VOLUME_PEERS":"monitor"}
q["reason_code"] = q.apply(reason_code, axis=1)
q["action"] = q["reason_code"].map(ACTION_FOR)
ev["reason_code"] = ev.apply(reason_code, axis=1)

FROZEN_MIX = {"AT_OR_ABOVE_VOLUME_PEERS":50.0, "CTR_BELOW_VOLUME_PEERS":18.9,
              "ZERO_CLICKS_ON_REAL_VOLUME":18.4, "CTR_FAR_BELOW_VOLUME_PEERS":12.8}
mix = (q["reason_code"].value_counts(normalize=True)*100).round(1).to_dict()
assert {k: round(v,1) for k,v in mix.items()} == FROZEN_MIX, f"playbook mix drifted: {mix}"

print("REASON CODE MIX (reproduced from the frozen ML-10 receipt)\n")
tbl = pd.DataFrame({"pages": q["reason_code"].value_counts(),
                    "share %": (q["reason_code"].value_counts(normalize=True)*100).round(1),
                    "under-capture rate": ev.groupby("reason_code")["y"].mean().round(4),
                    "action": pd.Series(ACTION_FOR)})
print(tbl.to_string())
print(f"\n  separation: ZERO_CLICKS {tbl.loc['ZERO_CLICKS_ON_REAL_VOLUME','under-capture rate']:.3f} "
      f"vs AT_OR_ABOVE {tbl.loc['AT_OR_ABOVE_VOLUME_PEERS','under-capture rate']:.3f} "
      f"= {tbl.loc['ZERO_CLICKS_ON_REAL_VOLUME','under-capture rate']/tbl.loc['AT_OR_ABOVE_VOLUME_PEERS','under-capture rate']:.1f}x")
print("\n  prune / merge / redirect recommendations: 0  <- nothing here measured whether a page")
print("  should exist. A low CTR is not evidence of no value.")

REASON CODE MIX (reproduced from the frozen ML-10 receipt)

                            pages  share %  under-capture rate               action
AT_OR_ABOVE_VOLUME_PEERS     5562     50.0              0.0722              monitor
CTR_BELOW_VOLUME_PEERS       2096     18.9              0.2964  review_intent_match
CTR_FAR_BELOW_VOLUME_PEERS   1418     12.8              0.5562   rewrite_title_meta
ZERO_CLICKS_ON_REAL_VOLUME   2043     18.4              0.5775   rewrite_title_meta

  separation: ZERO_CLICKS 0.578 vs AT_OR_ABOVE 0.072 = 8.0x

  prune / merge / redirect recommendations: 0  <- nothing here measured whether a page
  should exist. A low CTR is not evidence of no value.


### What must never be automated

| Never automate | Why |
|---|---|
| The title/meta rewrite itself | The system finds a *gap*. It has no evidence a rewrite closes it — and 42% close on their own. |
| Bulk-actioning the top N | p@50 = 0.92 means ~4 of 50 are wrong. At N = 200 it is ~36 wrong. Someone must look. |
| Delete / merge / redirect | Never measured. Not derivable from anything in this project. |
| Applying it to an unchecked new client | Client identity is recoverable from the features at AUC 0.94. A new client is out of distribution until tested. |
| Blending the rule and model scores | Two different jobs with different confidence levels. A blended number is unexplainable to a reviewer. |
| Publishing per-page rows externally | The release is pseudonymized for a reason. Aggregates only leave the repo. |

**The framing I will defend:** this is **decision-support**. It orders a reviewer's week and attaches a
reason to each row. It does not decide anything, it does not estimate value, and it has not yet been shown
to cause an improvement — because no holdout has been run. Recommendation 1 exists to fix that.

## 7. Reproducibility

In [12]:
OUT  = Path("work/outputs"); FIG = Path("work/figures")
DOCS = Path("docs");            SUB = Path("submission")
print("NOTEBOOKS - the chain every number comes from:\n")
for nb, what in [("w01_research_question.ipynb","the question and why it is answerable"),
                 ("w02_ml_task_framing.ipynb",  "task type, target, metric, leakage rules"),
                 ("w03_data_contract.ipynb",    "warehouse contract, windows, exclusions"),
                 ("w04_baseline_score.ipynb",   "the frozen four-line rule + p@K"),
                 ("w05_model.ipynb",            "4 models vs the frozen rule, GroupKFold"),
                 ("w06_validation_audit.ipynb", "split redesign + 3 leakage checks"),
                 ("w07_action_playbook.ipynb",  "the queue, reason codes, the 42% finding"),
                 ("capstone.ipynb",             "this notebook - the paper, re-run")]:
    p = Path("work/notebooks")/nb
    print(f"  {'OK ' if p.exists() else '-- '}{nb:32s} {what}")

print("\nRECEIPTS - every paper number traces to one of these:\n")
for p in sorted(OUT.glob("*.json")):
    m = json.loads(p.read_text())
    print(f"  {str(p):44s} {m.get('notebook','-')}")

print("\nFIGURES the paper embeds:")
for p in sorted(FIG.glob("*.png")) + sorted(OUT.glob("*.png")):
    print(f"  {str(p):52s} {p.stat().st_size:>8,} bytes")

print("\nDEPLOYMENT:")
for p, what in [(DOCS/"index.html", "the paper (self-contained, images inlined)"),
                (DOCS/".nojekyll", "tells GitHub Pages to serve files as-is"),
                (SUB/"paper_url.txt", "the deployed URL, one line")]:
    ok = "OK " if p.exists() else "-- "
    extra = f"({p.stat().st_size:,} bytes)" if p.exists() else "(not in this checkout)"
    print(f"  {ok}{str(p):26s} {what} {extra}")
if (SUB/"paper_url.txt").exists():
    print(f"\n  paper_url.txt contains: {(SUB/'paper_url.txt').read_text().strip()}")

print(f"\nENVIRONMENT: sklearn {sklearn.__version__} | pandas {pd.__version__} | "
      f"numpy {np.__version__} | seed {SEED} everywhere")

NOTEBOOKS - the chain every number comes from:

  OK w01_research_question.ipynb      the question and why it is answerable
  OK w02_ml_task_framing.ipynb        task type, target, metric, leakage rules
  OK w03_data_contract.ipynb          warehouse contract, windows, exclusions
  OK w04_baseline_score.ipynb         the frozen four-line rule + p@K
  OK w05_model.ipynb                  4 models vs the frozen rule, GroupKFold
  OK w06_validation_audit.ipynb       split redesign + 3 leakage checks
  OK w07_action_playbook.ipynb        the queue, reason codes, the 42% finding
  OK capstone.ipynb                   this notebook - the paper, re-run

RECEIPTS - every paper number traces to one of these:

  work/outputs/baseline_metrics.json           w04_baseline_score.ipynb
  work/outputs/model_metrics.json              w05_model.ipynb
  work/outputs/playbook_metrics.json           w07_action_playbook.ipynb
  work/outputs/validation_metrics.json         w06_validation_audit.ipynb

FIGURES t

In [13]:
# Final public-safety pass over everything this project publishes.
import re
BAD_PATTERNS = {
    "possible URL/domain": r"https?://(?!flyrank\.ai|github\.com/|colab\.research|huggingface\.co|[a-z0-9-]+\.github\.io)[\w.-]+\.[a-z]{2,}",
    "possible client name": r"\b(?:Inc|Ltd|LLC|GmbH|Corp)\b",
    "hf token": r"hf_[A-Za-z0-9]{20,}",
}
targets = [Path("docs/index.html")] if Path("docs/index.html").exists() else []
findings = []
for t in targets:
    txt = t.read_text(encoding="utf-8", errors="ignore")
    for label, pat in BAD_PATTERNS.items():
        hits = set(re.findall(pat, txt))
        if hits: findings.append((str(t), label, list(hits)[:5]))
print("FINAL PUBLIC-SAFETY PASS\n")
if targets:
    print(f"  scanned: {[str(t) for t in targets]}")
    print(f"  findings: {findings if findings else 'none - clean'}")
else:
    print("  docs/index.html not present in this checkout (it is committed in the repo).")
print(f"\n  identifiers used throughout: pseudonyms only, e.g. "
      f"{df['content_id'].iloc[0]} | {df['client_id'].iloc[0]}")
print("  no client names, domains, URLs, page titles or raw search queries exist in the")
print("  release, the notebooks, the exports or the paper.")

FINAL PUBLIC-SAFETY PASS

  scanned: ['docs/index.html']
  findings: none - clean

  identifiers used throughout: pseudonyms only, e.g. content_304f48230142 | client_f369cb89fc
  no client names, domains, URLs, page titles or raw search queries exist in the
  release, the notebooks, the exports or the paper.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — verified in the cell above
- [x] My claims use careful words: observed, measured, comparable, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### Capstone checklist

| Required | Where |
|---|---|
| Title + Abstract (question → method → result) | paper §Abstract |
| Introduction / problem statement | paper §1 · notebook §1 |
| Data — release, tables, windows, exclusions, public-safe | paper §2 · notebook §2 |
| Methodology — assumptions, features, label, baseline, validation, leakage | paper §3 · notebook §3 |
| Results — vs baseline on the same data, with charts | paper §4 · notebook §4 |
| Limitations & honest framing | paper §5 · notebook §5 |
| Ranked recommendations | paper §6 · notebook §6 |
| Reproducibility — notebooks and repo | paper §7 · notebook §7 |
| Acknowledgments & data credit (flyrank.ai) | paper §8 |
| Deployed at a public URL | `docs/index.html` via GitHub Pages |
| Exact URL in `submission/paper_url.txt` | one line, nothing else |

### The three sentences this work stands on

1. **Observed** — on 8,825 pages from 28 pseudonymized clients, a four-line arithmetic rule frozen before
   any model reaches p@50 = 0.92 against a 0.254 base rate.
2. **Measured** — four supervised models under `GroupKFold` on `client_id` do not beat it at K ≤ 50; every
   difference sits inside one fold-standard-deviation, and the model's defensible contribution is coverage
   of volume bands 0-2, where the rule's top 200 reaches 0 pages and the model's reaches 44.
3. **Decision-support** — 42% of flagged pages recover unaided within 30 days, so the output is a ranked
   review queue with reason codes and a mandatory holdout, carrying no value estimate and no automated
   action of any kind.

**Built on the FlyRank ML Internship dataset — [flyrank.ai](https://flyrank.ai).**